# OneData OmniBench · Lazizbek · 400 Complete Multisensory Robot Episodes

**Assigned tasks:** T2 Push & Slide; T4 Place in Container; T6 Tap & Knock; T7 Select & Sort. Exactly **100 accepted episodes per task**, with `60 train / 10 validation / 10 test_iid / 10 test_compositional / 10 test_visual_ood` per task. Thus **400 accepted per person** and **800 total** when merged.

**Source of design:** the verified `OneData_Omni_EE_OneEpisode_Colab_T4.ipynb` robot sensor/EE generator and `OneData_Omni_MOViC_10_Colab (1).ipynb` multimodal scene export schema from `BLTisandwich/OneData-MultiRep`.

**Deliverables from ONE physics episode:** three synchronized RGB-D + segmentation views; estimated optical flow; world-frame 3D point clouds; local object mesh and scene specs; robot joints, EE pose, next-commanded EE target and delta, gripper, object motion; force; synthetic contact/tactile proxy; event-aligned impact audio; actual TTS speech WAV and transcription; 8 programmatically grounded QAs with expert answers and evidence. All trainable targets are timestamped and last-frame masked.

**Important scientific limitations:** this is a **kinematic scripted IK expert**, not a physical demonstration of grasp-force closure. Pick/stack/container/drag/sort use **marked assisted attachments**; push may use marked assisted motion; button/tap are marked **virtual geometrical triggers**. Impact audio and tactile maps are **synthetic proxies**, not measured real sensors. This dataset supports prototyping EE-pose prediction and multimodal QA; do not claim physical closed-loop generalization solely from generated expert data.

**No Blender, no model training, no API required for ground truth.** T4 is optional; PyBullet uses CPU. Google Drive is required. **Do not upload the API key to GitHub.**

**Runtime:** Full generation may require several Colab sessions. Every validated episode is copied to Drive with checksums and an `_COMPLETE.json` marker; rerunning resumes without recomputing completed episodes. The Notebook first generates **one accepted episode from each of four task classes** before bulk generation. If a class cannot pass physics+schema checks, it stops rather than inventing success.


## 1. Install dependencies (runtime is Colab, CPU sufficient)

In [ ]:
import sys, subprocess, shutil
subprocess.run([sys.executable,'-m','pip','install','-q','pybullet>=3.2.6','numpy','scipy',
                'opencv-python-headless','imageio','imageio-ffmpeg','trimesh','requests','Pillow'],check=True)
if not (shutil.which('espeak-ng') or shutil.which('espeak')):
    subprocess.run(['apt-get','-qq','update'],check=True,stdout=subprocess.DEVNULL)
    subprocess.run(['apt-get','-qq','install','-y','espeak-ng'],check=True,stdout=subprocess.DEVNULL)
print('Dependencies ready. PyBullet uses CPU; a Colab T4 is NOT required for generation.')

## 2. Mount your own Google Drive, and (optional) paste your own language API credentials

The key stays in Colab RAM, is never written into the dataset, and is not included in the saved notebook. API only paraphrases **candidate questions**. Canonical questions and expert answers always come from the simulator. If you do not have an API, press Enter at the prompt.

In [ ]:
from google.colab import drive
from getpass import getpass
from pathlib import Path
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/OneData_OmniBench_v1'
API_CHAT_URL = 'https://api.openai.com/v1/chat/completions'  # edit if your provider uses another OpenAI-compatible endpoint
API_MODEL = 'gpt-4o-mini'  # edit to your provider's model name
API_KEY = getpass('Paste your own API key (Enter to skip API paraphrase): ').strip()
USE_API = bool(API_KEY)
print('Drive output root:', DRIVE_BASE)
print('API paraphrase:', 'enabled (candidate questions only)' if USE_API else 'disabled (canonical verified QA still produced)')

## 3. Shared generator implementation · versioned data protocol

Both teammates use **exactly the same code** below. Only the `owner` field changes. Do not manually change the episode schema or task IDs.

In [ ]:
"""OneData OmniBench 800: single-source, replayable PyBullet episode generator.
This module intentionally does NOT provide a learned controller: the commands are
scripted kinematic expert targets. Assisted grasp/push and proxy sensors are marked.
"""
from __future__ import annotations
import csv, hashlib, json, math, os, shutil, subprocess, time, wave
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np

DATASET_VERSION = 'onedata-omnibench-v1.0'
SPLITS = [('train', 60), ('val', 10), ('test_iid', 10),
          ('test_compositional', 10), ('test_visual_ood', 10)]
TASKS = {
    'T1': 'pick_place', 'T2': 'push_slide', 'T3': 'stack',
    'T4': 'place_container', 'T5': 'press_button', 'T6': 'tap_knock',
    'T7': 'select_sort', 'T8': 'drag_pull',
}
OWNERS = {'Lee': ['T1', 'T3', 'T5', 'T8'],
          'Lazizbek': ['T2', 'T4', 'T6', 'T7']}

@dataclass
class Config:
    owner: str = 'Lee'
    output_root: str = '/content/drive/MyDrive/OneData_OmniBench_v1'
    fps: int = 12
    physics_hz: int = 240
    duration_s: float = 8.0
    width: int = 192
    height: int = 144
    camera_names: tuple = ('front', 'overhead', 'side')
    sample_points: int = 256
    tactile_resolution: int = 16
    max_attempts: int = 14
    api_key: str = ''
    api_chat_url: str = ''
    api_model: str = ''
    enable_api: bool = False
    limit_per_task: int = 100
    save_demo_every: int = 10


def split_of(index):
    assert 0 <= index < 100
    offset = 0
    for name, count in SPLITS:
        if index < offset + count:
            return name
        offset += count
    raise AssertionError(index)


def manifest_for_owner(owner):
    assert owner in OWNERS, owner
    items = []
    for task in OWNERS[owner]:
        for i in range(100):
            items.append(dict(episode_id=f'{task}_{i:04d}', task_id=task,
                              task_name=TASKS[task], owner=owner,
                              split=split_of(i), attempt_seed=2026112300 + int(task[1:])*100000 + i*137))
    return items


def to_json(obj):
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.generic): return obj.item()
    if isinstance(obj, Path): return str(obj)
    if isinstance(obj, tuple): return list(obj)
    raise TypeError(type(obj))


def dump_json(path, obj):
    Path(path).write_text(json.dumps(obj, default=to_json, indent=2, ensure_ascii=False), encoding='utf-8')


def palette(split, rng):
    # Visual OOD uses disjoint colors/background from train/val/IID.
    if split == 'test_visual_ood':
        return {'target': [0.85,0.23,0.55,1], 'target_name':'magenta',
                'other':[0.20,0.85,0.75,1], 'other_name':'cyan',
                'background':[0.29,0.31,0.37,1], 'goal':[0.98,0.64,0.16,1],
                'goal_name':'orange'}
    colors = [[0.16,0.47,0.91,1], [0.93,0.48,0.19,1], [0.37,0.64,0.33,1]]
    color_idx=int(rng.integers(0,len(colors)))
    return {'target':colors[color_idx],'target_name':['blue','orange','green'][color_idx],
            'other':[0.68,0.63,0.58,1], 'other_name':'gray',
            'background':[0.78,0.80,0.83,1],
            'goal':[0.11,0.80,0.45,1],'goal_name':'green'}


def sample_spec(row, seed):
    rng = np.random.default_rng(seed)
    tid = row['task_id']; split=row['split']
    bx = float(rng.uniform(0.49,0.56)); by=float(rng.uniform(-0.19,-0.10))
    gx = float(rng.uniform(0.42,0.59)); gy=float(rng.uniform(0.13,0.23))
    if split == 'test_compositional':
        # Target on a held-out side of the start position.
        gx=float(rng.uniform(0.40,0.46)); gy=float(rng.uniform(-0.25,-0.15))
    if tid in ('T2','T8'):
        gx=bx; gy=float(rng.uniform(0.10,0.18))
    if tid in ('T5','T6'):
        gx=bx;gy=by
    edge=float(rng.uniform(0.045,0.057))
    return dict(**row,seed=int(seed),initial_xyz=[bx,by,edge/2],
                target_xyz=[gx,gy,edge/2],cube_edge_m=edge,
                distractor_xyz=[[0.43,-0.23,0.03],[0.57,0.19,0.03]],
                style=palette(split,rng),
                task_physics={'gravity_m_s2':-9.81,'lateral_friction':1.4,'restitution':0.03},
                coordinate_frame='PyBullet_world_Z_up',
                object_asset='procedural_box_mesh_with_separate_physics_proxy',
                ee_command_definition='world-frame Cartesian target at 12Hz; expert is kinematic IK teacher')


def make_instruction(spec):
    tid=spec['task_id']; style=spec['style']
    color=style['target_name'];goal=style['goal_name']
    other=style['other_name']
    phrases={
        'T1':f'Pick up the {color} cube and place it on the {goal} target marker.',
        'T2':f'Push the {color} cube across the table to the {goal} marker.',
        'T3':f'Stack the {color} cube on top of the {other} stationary support.',
        'T4':f'Pick up the {color} cube and drop it inside the {other} open container.',
        'T5':f'Press the {color} button until it activates.',
        'T6':f'Tap the {color} cube once and then retract the end-effector.',
        'T7':f'Select the {color} cube, not the {other} distractors, '
             f'and move it onto the {goal} target marker.',
        'T8':f'Grip the {color} cube and drag it over the tabletop to the {goal} marker.'}
    return phrases[tid]


def waypoints(spec):
    bx,by,bz=spec['initial_xyz']; gx,gy,_=spec['target_xyz'];
    tid=spec['task_id']; zc=.09
    def wp(t,x,y,z,g,phase):return (float(t),[float(x),float(y),float(z)],float(g),phase)
    # 8s total, per-finger opening in meters. All commanded poses are world frame.
    if tid in ('T1','T3','T4','T7','T8'):
        release_z=.09
        if tid=='T3':release_z=.16 # above support cube
        if tid=='T4':release_z=.19 # above open container rim
        if tid=='T8':release_z=.09 # keep object touching table
        lift_z=.10 if tid=='T8' else .28
        return [wp(0,bx-.09,by,.31,.04,'approach'),wp(1.0,bx,by,.22,.04,'approach'),
                wp(2.0,bx,by,zc,.04,'descend'),wp(2.55,bx,by,zc,0,'grasp'),
                wp(3.0,bx,by,zc,0,'grasp_hold'),wp(4.0,bx,by,lift_z,0,'lift'),
                wp(5.4,gx,gy,lift_z,0,'transfer'),wp(6.55,gx,gy,release_z,0,'lower'),
                wp(6.85,gx,gy,release_z,.04,'release'),wp(7.75,gx,gy,.30,.04,'retreat'),
                wp(8.0,gx,gy,.30,.04,'done')]
    if tid=='T2':
        return [wp(0,bx,by-.15,.25,.04,'approach'),wp(1.5,bx,by-.095,.07,.04,'align'),
                wp(2.3,bx,by-.045,.06,.04,'contact'),wp(5.6,gx,gy-.04,.06,.04,'push'),
                wp(6.4,gx,gy-.04,.06,.04,'settle'),wp(7.6,gx,gy-.08,.28,.04,'retreat'),
                wp(8,gx,gy-.08,.28,.04,'done')]
    if tid in ('T5','T6'):
        # Contact events use geometric tool proximity; verify episode outcome separately.
        contact_z=.048 if tid=='T5' else .052
        return [wp(0,bx-.08,by,.27,.04,'approach'),wp(1.3,bx,by,.20,.04,'align'),
                wp(2.6,bx,by,.085,.04,'descend'),wp(3.2,bx,by,contact_z,.04,'press' if tid=='T5' else 'tap'),
                wp(4.2,bx,by,contact_z,.04,'hold' if tid=='T5' else 'tap_hold'),
                wp(5.1,bx,by,.23,.04,'retreat'),wp(7.7,bx-.08,by,.27,.04,'retreat'),
                wp(8,bx-.08,by,.27,.04,'done')]
    raise KeyError(tid)


def pybullet_episode(spec,cfg,out):
    import pybullet as pb, pybullet_data
    from scipy.io import wavfile
    import imageio.v2 as imageio
    import trimesh
    out=Path(out); out.mkdir(parents=True,exist_ok=True)
    rng=np.random.default_rng(spec['seed']); tid=spec['task_id']; duration=cfg.duration_s
    assert cfg.physics_hz % cfg.fps == 0 and cfg.fps >= 2
    dt=1/cfg.physics_hz; frames=int(round(duration*cfg.fps)); sub=cfg.physics_hz//cfg.fps
    cid=pb.connect(pb.DIRECT)
    try:
        pb.setAdditionalSearchPath(pybullet_data.getDataPath());pb.setTimeStep(dt)
        pb.setGravity(0,0,-9.81);pb.setPhysicsEngineParameter(numSolverIterations=80)
        floor=pb.loadURDF('plane.urdf',[0,0,-.002]); pb.changeVisualShape(floor,-1,rgbaColor=spec['style']['background'])
        robot=pb.loadURDF('franka_panda/panda.urdf',[0,0,0],useFixedBase=True)
        assert pb.getNumJoints(robot)>=12
        arm_j=list(range(7)); fingers=[9,10]; ee_link=11
        edge=spec['cube_edge_m']; bx,by,bz=spec['initial_xyz'];gx,gy,_=spec['target_xyz']
        def add_box(pos,size,color,mass=0.0):
            c=pb.createCollisionShape(pb.GEOM_BOX,halfExtents=[v/2 for v in size]);
            v=pb.createVisualShape(pb.GEOM_BOX,halfExtents=[v/2 for v in size],rgbaColor=color)
            body=pb.createMultiBody(baseMass=mass,baseCollisionShapeIndex=c,baseVisualShapeIndex=v,basePosition=pos)
            pb.changeDynamics(body,-1,lateralFriction=1.4,restitution=.03)
            return body
        cube=add_box([bx,by,bz],[edge]*3,spec['style']['target'],mass=(0 if tid in ('T5','T6') else .09))
        static={}; dynamic={'target':cube}
        # Goal marker does not collide with dynamics.
        marker=pb.createVisualShape(pb.GEOM_CYLINDER,radius=.055,length=.003,rgbaColor=spec['style']['goal'])
        goalbody=pb.createMultiBody(baseMass=0,baseVisualShapeIndex=marker,basePosition=[gx,gy,.002])
        static['goal']=goalbody
        if tid=='T3':
            # Stable support height exactly 0.054m; goal at support center.
            support=add_box([gx,gy,.027],[.075,.075,.054],spec['style']['other'],0)
            static['support']=support
        elif tid=='T4':
            # Open container: separate solid walls with no floor collision ambiguity.
            w=.14; h=.075; thick=.009
            static['bin_base']=add_box([gx,gy,.005],[w,w,.01],[.55,.58,.61,1])
            for name,dx,dy,s in [('north',0,w/2,[w,thick,h]),('south',0,-w/2,[w,thick,h]),
                                  ('east',w/2,0,[thick,w,h]),('west',-w/2,0,[thick,w,h])]:
                static['bin_'+name]=add_box([gx+dx,gy+dy,h/2+.005],s,[.52,.62,.69,1])
        elif tid=='T7':
            for i,pos in enumerate(spec['distractor_xyz']):
                static['distractor_'+str(i)]=add_box(pos,[.06]*3,spec['style']['other'],0)
        elif tid in ('T5','T6'):
            # The primary static object serves as the virtual button/tap target;
            # do not create overlapping collision boxes at the same position.
            pass
        # Camera transform records are used for RGB-D unprojection.
        cams={
          'front':([1.21,-1.05,.88],[.42,-.01,.14]),
          'overhead':([.65,0,1.24],[.48,0,.04]),
          'side':([.91,.92,.68],[.46,0,.15]),
        }
        near=.04;far=3.0; W=cfg.width;H=cfg.height
        projections={};viewmats={};cameras={}
        for cname in cfg.camera_names:
            eye,target=cams[cname]; v=pb.computeViewMatrix(eye,target,[0,0,1]);
            proj=pb.computeProjectionMatrixFOV(54,W/H,near,far)
            cameras[cname]={'eye':eye,'target':target,'view_col_major':list(v),
                            'projection_col_major':list(proj),'near_m':near,'far_m':far,
                            'width':W,'height':H}
            viewmats[cname]=v;projections[cname]=proj
        qquat=pb.getQuaternionFromEuler([math.pi,0,math.pi/4])
        wps=waypoints(spec); wt=np.asarray([x[0] for x in wps]);
        lower=[pb.getJointInfo(robot,j)[8] for j in arm_j];
        upper=[pb.getJointInfo(robot,j)[9] for j in arm_j]
        ranges=[u-l for l,u in zip(lower,upper)];rest=[0,-.35,0,-2.2,0,1.9,.79]
        qplan=[];errors=[]
        for t,xyz,_,phase in wps:
            q=pb.calculateInverseKinematics(robot,ee_link,xyz,qquat,
                lowerLimits=lower,upperLimits=upper,jointRanges=ranges,
                restPoses=rest,maxNumIterations=180,residualThreshold=1e-5)
            q=np.asarray(q[:7]);qplan.append(q)
            for j,v in zip(arm_j,q): pb.resetJointState(robot,j,float(v))
            actual=np.asarray(pb.getLinkState(robot,ee_link,computeForwardKinematics=True)[4])
            errors.append(float(np.linalg.norm(actual-np.asarray(xyz))))
        if max(errors)>.14: raise RuntimeError('IK waypoint error too large '+str(max(errors)))
        def plan_at(t):
            k=int(np.clip(np.searchsorted(wt,t,side='right')-1,0,len(wps)-2))
            alpha=float(np.clip((t-wt[k])/(wt[k+1]-wt[k]),0,1));alpha=alpha*alpha*(3-2*alpha)
            q=(1-alpha)*qplan[k]+alpha*qplan[k+1]
            pos=(1-alpha)*np.asarray(wps[k][1])+alpha*np.asarray(wps[k+1][1]);
            g=(1-alpha)*wps[k][2]+alpha*wps[k+1][2]
            return q,pos,float(g),wps[k+1][3]
        for j,q in zip(arm_j,qplan[0]):pb.resetJointState(robot,j,float(q))
        for j in fingers:pb.resetJointState(robot,j,.04)
        # Kinematic expert attachment is not a physical force-closure grasp.
        held=False; relative=None; proxy_push=False; prev_ee=None; events=[]; event_once=set();
        motion=[]; vision={n:{'rgb':[],'depth_mm':[],'seg':[]} for n in cfg.camera_names}
        fast={'time_s':[],'command_xyz_world_m':[],'actual_ee_xyz_world_m':[],
              'object_xyz_world_m':[],'gripper_command_m':[],
              'joint_position_rad':[],'robot_object_contact_N':[]}
        physical={k:[] for k in ['t','q','dq','ee_pos','ee_quat','cube_pos','cube_quat','cube_vel','contact_N','held']}
        grasp_time=2.80;release_time=6.75
        last_ee=None; last_q=None
        for step in range(int(round(duration*cfg.physics_hz))):
            t=(step+1)*dt; q,pos,g,phase=plan_at(t)
            for j,v in zip(arm_j,q): pb.resetJointState(robot,j,float(v))
            for j in fingers:pb.resetJointState(robot,j,float(np.clip(g,0,.04)))
            ee=pb.getLinkState(robot,ee_link,computeForwardKinematics=True)
            ee_xyz=np.asarray(ee[4]);ee_quat=ee[5]
            old_cube_pos=np.asarray(pb.getBasePositionAndOrientation(cube)[0])
            new_held=(tid in ('T1','T3','T4','T7','T8')) and (grasp_time<=t<release_time)
            if new_held and not held:
                invp,invq=pb.invertTransform(ee[4],ee[5]);cp,cq=pb.getBasePositionAndOrientation(cube)
                relative=pb.multiplyTransforms(invp,invq,cp,cq);held=True
                if 'grasp' not in event_once:
                    events.append({'name':'assisted_grasp_start','time_s':t,'provenance':'scripted_attachment'});event_once.add('grasp')
            if held and not new_held:
                held=False; events.append({'name':'assisted_release','time_s':t,'provenance':'scripted_attachment'})
            if held:
                cp,cq=pb.multiplyTransforms(ee[4],ee[5],relative[0],relative[1]);
                pb.resetBasePositionAndOrientation(cube,cp,cq);pb.resetBaseVelocity(cube,[0,0,0],[0,0,0])
            if tid=='T2' and phase=='push' and last_ee is not None:
                # Explicit assist, activated only in a spatial contact corridor.
                if abs(ee_xyz[0]-old_cube_pos[0]) < .09 and ee_xyz[1] < old_cube_pos[1]+.025:
                    delta=float(np.clip(ee_xyz[1]-last_ee[1],0,.012))
                    cp=[float(old_cube_pos[0]),float(min(gy,old_cube_pos[1]+delta)),float(bz)]
                    pb.resetBasePositionAndOrientation(cube,cp,[0,0,0,1]);
                    pb.resetBaseVelocity(cube,[0,0,0],[0,0,0]);proxy_push=True
            pb.stepSimulation()
            if held:
                fresh=pb.getLinkState(robot,ee_link,computeForwardKinematics=True)
                cp,cq=pb.multiplyTransforms(fresh[4],fresh[5],relative[0],relative[1]);
                pb.resetBasePositionAndOrientation(cube,cp,cq);pb.resetBaseVelocity(cube,[0,0,0],[0,0,0])
            last_ee=ee_xyz.copy()
            fast['time_s'].append(t)
            fast['command_xyz_world_m'].append(np.asarray(pos,np.float32))
            fast['actual_ee_xyz_world_m'].append(ee_xyz.astype(np.float32))
            fast['object_xyz_world_m'].append(np.asarray(pb.getBasePositionAndOrientation(cube)[0],np.float32))
            fast['gripper_command_m'].append(g)
            fast['joint_position_rad'].append(q.astype(np.float32))
            fast['robot_object_contact_N'].append(float(sum(cp[9] for cp in pb.getContactPoints(bodyA=robot,bodyB=cube))))
            if step % sub==0:
                contact_N=float(sum(cp[9] for cp in pb.getContactPoints(bodyA=robot,bodyB=cube)))
                if tid in ('T5','T6'):
                    # Button/tap interaction is a geometry-triggered virtual event.
                    # It is not misreported as measured force or a physical button joint.
                    if (phase in ('press','hold','tap','tap_hold')
                         and np.linalg.norm(ee_xyz[:2]-np.array([bx,by]))<.075
                         and ee_xyz[2] < .12):
                        key='button_activated' if tid=='T5' else 'tap_detected'
                        if key not in event_once:
                            events.append({'name':key,'time_s':t,'provenance':'geometric_virtual_switch'});event_once.add(key)
                if contact_N>.5 and 'contact' not in event_once:
                    events.append({'name':'physical_contact_detected','time_s':t,'provenance':'pybullet_contact'});event_once.add('contact')
                pose,rot=pb.getBasePositionAndOrientation(cube)
                vel,avel=pb.getBaseVelocity(cube)
                qnow=np.array([pb.getJointState(robot,j)[0] for j in arm_j])
                dqnow=(qnow-last_q)/(sub*dt) if last_q is not None else np.zeros(7)
                last_q=qnow.copy()
                physical['t'].append(t);physical['q'].append(qnow);physical['dq'].append(dqnow)
                physical['ee_pos'].append(ee_xyz);physical['ee_quat'].append(ee_quat)
                physical['cube_pos'].append(pose);physical['cube_quat'].append(rot)
                physical['cube_vel'].append(vel);physical['contact_N'].append(contact_N)
                physical['held'].append(held)
                motion.append(dict(phase=phase,command_pos=pos,command_quat=qquat,
                    gripper_cmd=g,ee_pos=ee_xyz,ee_quat=ee_quat,
                    robot_q=qnow,finger_positions=[pb.getJointState(robot,j)[0] for j in fingers]))
                for cname in cfg.camera_names:
                    img=pb.getCameraImage(W,H,viewmats[cname],projections[cname],
                                          renderer=pb.ER_TINY_RENDERER,
                                          flags=pb.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX)
                    rgb=np.asarray(img[2],dtype=np.uint8).reshape(H,W,4)[:,:,:3]
                    zb=np.asarray(img[3],dtype=np.float32).reshape(H,W)
                    dm=far*near/(far-(far-near)*zb)
                    segmentation=np.asarray(img[4],dtype=np.int32).reshape(H,W)
                    vision[cname]['rgb'].append(rgb.copy());
                    vision[cname]['depth_mm'].append(np.clip(dm*1000,0,65535).astype(np.uint16));
                    vision[cname]['seg'].append(segmentation)
        assert len(motion)==frames,(len(motion),frames)
        np.savez_compressed(out/'physics_240hz.npz',**{key:np.asarray(vals,dtype=np.float32) for key,vals in fast.items()})
        for name in physical:physical[name]=np.asarray(physical[name])
        for cname in cfg.camera_names:
            for ch in ('rgb','depth_mm','seg'):vision[cname][ch]=np.asarray(vision[cname][ch])
        M=lambda field:np.asarray([m[field] for m in motion])
        ts=physical['t'].astype(np.float64)
        # Expert commanded EE targets vs actually observed EE poses are distinct.
        cmds=M('command_pos').astype(np.float32)
        actual=M('ee_pos').astype(np.float32)
        target_g=M('gripper_cmd').astype(np.float32)
        # Critical causal label: o_t predicts expert command for the NEXT time step.
        # Using cmds[t] - observed[t] would be nearly zero under kinematic IK!
        next_cmd=np.vstack((cmds[1:],cmds[-1:])).astype(np.float32)
        next_grip=np.r_[target_g[1:],target_g[-1]].astype(np.float32)
        actions=dict(timestamp_s=ts,commanded_ee_target_world_m=cmds,
                     expert_next_ee_target_world_m=next_cmd,
                     expert_next_ee_delta_from_current_world_m=(next_cmd-actual).astype(np.float32),
                     commanded_ee_quaternion_world_xyzw=M('command_quat').astype(np.float32),
                     executed_ee_position_world_m=actual,
                     executed_ee_quaternion_world_xyzw=M('ee_quat').astype(np.float32),
                     commanded_delta_world_m=(cmds-actual).astype(np.float32),
                     executed_next_ee_delta_world_m=np.vstack((np.diff(actual,axis=0),np.zeros((1,3)))).astype(np.float32),
                     gripper_command_m=target_g,
                     gripper_next_command_m=next_grip,
                     action_valid=np.r_[np.ones(frames-1,bool),False],
                     expert_phase=np.array([m['phase'] for m in motion]))
        np.savez_compressed(out/'actions.npz',**actions)
        # All physics-relevant replay data and parameters; torque is intentionally absent
        # from kinematic joint-state reset (it would be meaningless to fabricate).
        ee_vel=np.vstack((np.zeros((1,3)),np.diff(actual,axis=0)*cfg.fps)).astype(np.float32)
        object_vel_fd=np.vstack((np.zeros((1,3)),np.diff(physical['cube_pos'],axis=0)*cfg.fps)).astype(np.float32)
        button_state=np.zeros(frames,dtype=np.uint8); tap_state=np.zeros(frames,dtype=np.uint8)
        for event in events:
            if event['name']=='button_activated':button_state[ts>=event['time_s']]=1
            if event['name']=='tap_detected':tap_state[ts>=event['time_s']]=1
        np.savez_compressed(out/'robot_object_state.npz',
            timestamp_s=ts, joint_position_rad=physical['q'].astype(np.float32),
            joint_velocity_rad_s=physical['dq'].astype(np.float32),
            finger_position_m=M('finger_positions').astype(np.float32),
            ee_position_world_m=actual,
            ee_velocity_fd_world_m_s=ee_vel,
            ee_quaternion_world_xyzw=M('ee_quat').astype(np.float32),
            object_position_world_m=physical['cube_pos'].astype(np.float32),
            object_quaternion_world_xyzw=physical['cube_quat'].astype(np.float32),
            object_velocity_reported_pybullet_world_m_s=physical['cube_vel'].astype(np.float32),
            object_velocity_finite_difference_world_m_s=object_vel_fd,
            virtual_button_active=button_state,virtual_tap_detected=tap_state,
            robot_object_physical_contact_N=physical['contact_N'].astype(np.float32),
            scripted_attachment_active=physical['held'])
        # Use all camera streams at the same timestamps; preserve raw images in NPZ.
        visdump={'timestamp_s':ts}
        for cname in cfg.camera_names:
            visdump[cname+'_rgb']=vision[cname]['rgb']
            visdump[cname+'_depth_mm']=vision[cname]['depth_mm']
            visdump[cname+'_segmentation_int32']=vision[cname]['seg']
        # Optical flow is derived from RGB, not physics ground truth.
        import cv2
        first=vision[cfg.camera_names[0]]['rgb']
        flow=np.zeros((frames,H,W,2),dtype=np.float16)
        for k in range(1,frames):
            prev=cv2.cvtColor(first[k-1],cv2.COLOR_RGB2GRAY)
            cur=cv2.cvtColor(first[k],cv2.COLOR_RGB2GRAY)
            flow[k]=cv2.calcOpticalFlowFarneback(prev,cur,None,.5,2,13,2,5,1.1,0).astype(np.float16)
        visdump['front_optical_flow_rgb_estimated_px']=flow
        np.savez_compressed(out/'multiview_rgb_depth_seg_flow.npz',**visdump)
        # Dense point clouds are reconstructible from depth + full calibration.
        # Store per-frame sampled world-frame 3D to exercise 3D encoders directly.
        pcs={};valids={}
        xx,yy=np.meshgrid(np.arange(W),np.arange(H));xclip=2*(xx+.5)/W-1
        yclip=1-2*(yy+.5)/H
        for cname in cfg.camera_names:
            V=np.asarray(viewmats[cname]).reshape(4,4,order='F')
            P=np.asarray(projections[cname]).reshape(4,4,order='F')
            inv=np.linalg.inv(P@V)
            d=vision[cname]['depth_mm'].astype(np.float64)/1000
            seg=vision[cname]['seg'];pc=[];mask=[]
            for k in range(frames):
                ids=np.where(seg[k]<0,-1,seg[k]&((1<<24)-1))
                ys,xs=np.where((ids>=0)&(d[k]<far-.05)&(d[k]>near))
                if len(xs)==0:
                    pc.append(np.zeros((cfg.sample_points,3),np.float32));mask.append(False);continue
                ix=rng.choice(len(xs),cfg.sample_points,replace=len(xs)<cfg.sample_points)
                px=xs[ix];py=ys[ix];z=d[k,py,px];zb=(far-far*near/z)/(far-near)
                clip=np.stack([xclip[py,px],yclip[py,px],2*zb-1,np.ones_like(z)],axis=-1)
                world=clip@inv.T;world=world[:,:3]/world[:,3:4]
                pc.append(world.astype(np.float32));mask.append(True)
            pcs[cname+'_world_xyz_m']=np.stack(pc)
            valids[cname+'_valid']=np.asarray(mask)
        np.savez_compressed(out/'pointcloud_world.npz',timestamp_s=ts,**pcs,**valids)
        # Frame-wise contact proxy is *not* independent measured tactile sensing.
        YY,XX=np.mgrid[0:cfg.tactile_resolution,0:cfg.tactile_resolution]
        pad=np.exp(-((XX-7.5)**2+(YY-7.5)**2)/(2*2**2))
        tactile=np.zeros((frames,2,cfg.tactile_resolution,cfg.tactile_resolution),np.float32)
        for k in range(frames):
            proxy=5. if physical['held'][k] else float(physical['contact_N'][k])
            if tid in ('T5','T6') and any(e['name'] in ('button_activated','tap_detected') and abs(e['time_s']-ts[k])<.15 for e in events):
                proxy=max(proxy,3.)
            tactile[k]=pad[None]*proxy
        np.savez_compressed(out/'tactile_proxy.npz',timestamp_s=ts,
            pressure_2pads_PROXY=tactile,source=np.asarray('kinematic_grip_or_contact_proxy'))
        # Audio is synthesized at recorded event timestamps, not measured sound.
        sr=16000;sfx=np.zeros(int(duration*sr),np.float32)
        for i,e in enumerate(events):
            start=int(e['time_s']*sr);n=min(int(.18*sr),len(sfx)-start)
            if n<=0:continue
            tt=np.arange(n)/sr;noise=np.random.default_rng(400+spec['seed']+i).standard_normal(n)
            signal=(np.sin(2*np.pi*(390+110*i)*tt)+.1*noise)*np.exp(-23*tt)
            sfx[start:start+n]+=(.17*signal).astype(np.float32)
        wavfile.write(str(out/'impact_audio_SYNTHETIC.wav'),sr,np.int16(np.clip(sfx,-1,1)*32767))
        instruction=make_instruction(spec)
        exe=shutil.which('espeak-ng') or shutil.which('espeak')
        if not exe:raise RuntimeError('Real speech synthesis unavailable. Install espeak-ng; do not substitute silence.')
        cp=subprocess.run([exe,'-s','139','-w',str(out/'spoken_instruction.wav'),instruction],
                          capture_output=True,text=True)
        if cp.returncode or (out/'spoken_instruction.wav').stat().st_size<512:
            raise RuntimeError('espeak-ng failed: '+cp.stderr[-500:])
        # Mesh target object in LOCAL frame; URDF robot mesh refs from pybullet_data.
        meshpath=out/'target_cube_local.obj'
        trimesh.creation.box(extents=[edge]*3).export(meshpath)
        # Video is a lossy preview only; the original RGB arrays are preserved.
        with imageio.get_writer(str(out/'expert_preview.mp4'),fps=cfg.fps,codec='libx264',quality=6,
                                macro_block_size=16) as writer:
            for im in first:writer.append_data(im)
        pos_end=np.asarray(pb.getBasePositionAndOrientation(cube)[0])
        rel=float(np.linalg.norm(pos_end[:2]-np.array([gx,gy])))
        special_names={e['name'] for e in events}
        if tid=='T3':
            success=rel<.045 and pos_end[2]>.075
        elif tid=='T4':
            success=rel<.047 and pos_end[2]<.085
        elif tid=='T5':success='button_activated' in special_names
        elif tid=='T6':success='tap_detected' in special_names
        else:success=rel<.06 and pos_end[2]<.085
        # Scene and camera invariants, with full source provenance.
        sim_meta={
            'engine':'PyBullet DIRECT + TinyRenderer','physics_hz':cfg.physics_hz,
            'fps':cfg.fps,'duration_s':duration,'num_frames':frames,
            'camera_calibration':cameras,'object_body_ids':{'robot':robot,'target':cube,**static},
            'robot_urdf':'pybullet_data/franka_panda/panda.urdf',
            'robot_joint_ids':arm_j,'finger_joint_ids':fingers,'ee_link_id':ee_link,
            'mesh':'target_cube_local.obj','object_size_m':edge,
            'waypoints':wps,'max_ik_position_error_m':max(errors),
            'assisted_grasp':tid in ('T1','T3','T4','T7','T8'),
            'assisted_push':tid=='T2' and proxy_push,
            'virtual_contact_state_machine':tid in ('T5','T6'),
            'contact_force_source':'pybullet getContactPoints; does not include attachment forces',
            'tactile_source':'synthetic proxy generated from grip state/contact force',
            'impact_audio_source':'synthetic tones aligned to recorded events',
            'physics_limitations':'Kinematic joint reset; not dynamic actuation/real force closure',
            'success_criteria':'task-specific simulator measured final state or virtual button/tap event'
        }
        dump_json(out/'scene_and_replay.json',sim_meta)
        dump_json(out/'events.json',events)
        dump_json(out/'task.json',dict(spec, instruction=instruction,
                                      success=bool(success),final_object_position_world_m=pos_end.tolist(),
                                      final_goal_distance_xy_m=rel))
        return {'success':bool(success),'events':events,'duration_s':duration,
                'num_frames':frames,'final_error_xy_m':rel,'final_pos':pos_end.tolist(),
                'instruction':instruction,'max_ik_error_m':max(errors),
                'actual_ee_first':actual[0].tolist(),'actual_ee_last':actual[-1].tolist(),
                'phase_list':[m['phase'] for m in motion],
                'contact_peak_N':float(np.max(physical['contact_N'])),
                'grasp_assisted':bool(tid in ('T1','T3','T4','T7','T8'))}
    finally:
        pb.disconnect(cid)


def canonical_qa(spec,episode_result):
    """Answers are determined from saved simulator state, not guessed by an LLM."""
    sp=spec;result=episode_result; start=sp['initial_xyz']; goal=sp['target_xyz'];
    success=result['success'];events=result['events']; phases=result['phase_list'];
    firstevent=events[0]['time_s'] if events else None
    def qa(cat,q,a,typ,mods,source,interval=None):
        return {'category':cat,'canonical_question':q,'question':q,
                'expert_answer':a,'answer_type':typ,'evidence_modalities':mods,
                'answer_source':source,'evidence_time_range_s':interval,
                'api_paraphrase':None,'api_paraphrase_verified':False}
    name=sp['task_name'].replace('_',' ')
    qas=[
       qa('visual', 'What manipulation task is shown in this episode?',name,'string',
          ['video','task_instruction'],'task_spec'),
       qa('spatial','Is the target x-coordinate smaller than the start x-coordinate?',
          'yes' if goal[0]<start[0] else 'no','yes_no',['depth','pointcloud','object_pose'], 'scene_spec'),
       qa('temporal','At what time (in seconds) did the first recorded interaction event occur? Answer none if no event occurred.',
          f"{firstevent:.2f}" if firstevent is not None else 'none',
          'decimal_or_none',['events','robot_timeseries','audio'],'recorded_event_timestamps'),
       qa('contact','Was the gripper commanded to close in this episode?',
          'yes' if result['grasp_assisted'] else 'no','yes_no',
          ['robot_state','gripper'],'recorded_gripper_commands'),
       qa('audio','How many recorded event cues drive the synthetic impact audio?',
          str(len(events)),'integer',['impact_audio','events'],'event_log'),
       qa('ee_action','Does the end-effector finish higher than it starts?',
          'yes' if result['actual_ee_last'][2]>result['actual_ee_first'][2] else 'no',
          'yes_no',['ee_pose','time_series'],'measured_ee_poses'),
       qa('outcome','Does the manipulated object finish higher than its initial height?',
          'yes' if result['final_pos'][2]>start[2]+.03 else 'no',
          'yes_no',['object_pose','video'],'actual_object_start_and_end_states'),
       qa('cross_modal','Does the episode contain a nonzero PyBullet robot-object contact force?',
          'yes' if result['contact_peak_N']>0 else 'no','yes_no',
          ['force','robot_state','video'],'pybullet_contact_forces')]
    return qas


def api_paraphrase_batch(spec,qas,cfg):
    """One optional API call per episode, never rewrites simulator oracle gold labels."""
    if not (cfg.enable_api and cfg.api_key and cfg.api_chat_url and cfg.api_model):
        return qas
    import requests
    prompts=[{'id':i,'question':q['canonical_question'],
              'gold_answer':q['expert_answer']} for i,q in enumerate(qas)]
    payload={'model':cfg.api_model,'temperature':.3,'messages':[
      {'role':'system','content':(
        'Rewrite exactly 8 English robotics questions, preserving exact semantics, '
        'time scope, referents and answer. Return ONLY JSON {"questions":[8 strings]} '
        'in same order, with no gold answers in output. Do not add information.')},
      {'role':'user','content':json.dumps({'task':spec['task_name'],
                                          'items':prompts})}]}
    try:
        r=requests.post(cfg.api_chat_url,
            headers={'Authorization':'Bearer '+cfg.api_key,
                     'Content-Type':'application/json'},json=payload,timeout=45)
        r.raise_for_status()
        txt=r.json()['choices'][0]['message']['content'].strip()
        if txt.startswith('```'):
            txt=txt.strip('`').replace('json\n','',1).strip()
        data=json.loads(txt)
        rewrites=data['questions']
        if len(rewrites)==len(qas):
            for q,v in zip(qas,rewrites):
                if isinstance(v,str) and 10<len(v)<240 and '?' in v and '\n' not in v:
                    q['api_paraphrase']=v
                    q['api_paraphrase_verified']=False
    except Exception as exc:
        # API failure does not invalidate independently verified oracle questions.
        for q in qas:q['api_error']=str(exc)[:150]
    return qas


def validate_episode(path,spec,result,cfg):
    pth=Path(path)
    required=['actions.npz','robot_object_state.npz','physics_240hz.npz','multiview_rgb_depth_seg_flow.npz',
              'pointcloud_world.npz','tactile_proxy.npz','spoken_instruction.wav',
              'impact_audio_SYNTHETIC.wav','scene_and_replay.json','task.json','events.json',
              'qa.jsonl','expert_preview.mp4','target_cube_local.obj']
    checks={'files_present':all((pth/f).is_file() and (pth/f).stat().st_size>64 for f in required),
            'task_success':bool(result['success'])}
    a=np.load(pth/'actions.npz');v=np.load(pth/'multiview_rgb_depth_seg_flow.npz')
    s=np.load(pth/'robot_object_state.npz');pc=np.load(pth/'pointcloud_world.npz')
    tactile=np.load(pth/'tactile_proxy.npz'); fast=np.load(pth/'physics_240hz.npz')
    T=int(cfg.duration_s*cfg.fps); t=a['timestamp_s'];
    checks['aligned_timestamps']=bool(np.allclose(t,v['timestamp_s']) and
       np.allclose(t,s['timestamp_s']) and np.allclose(t,pc['timestamp_s']) and
       np.allclose(t,tactile['timestamp_s']))
    checks['frame_count']=bool(len(t)==T)
    checks['physics_length']=bool(len(fast['time_s'])==int(cfg.physics_hz*cfg.duration_s))
    checks['finite_actions']=bool(np.isfinite(a['commanded_ee_target_world_m']).all())
    checks['last_mask']=bool(np.all(a['action_valid'][:-1]) and not a['action_valid'][-1])
    checks['next_actual_delta']=bool(np.allclose(a['executed_next_ee_delta_world_m'][:-1],
                   np.diff(a['executed_ee_position_world_m'],axis=0),atol=1e-5))
    checks['causal_expert_action']=bool(np.allclose(a['expert_next_ee_target_world_m'][:-1],
                   a['commanded_ee_target_world_m'][1:],atol=1e-5) and
                   np.allclose(a['expert_next_ee_delta_from_current_world_m'][:-1],
                   a['commanded_ee_target_world_m'][1:]-a['executed_ee_position_world_m'][:-1],atol=1e-5))
    checks['all_camera_modalities']=all(v[k+'_rgb'].shape==(T,cfg.height,cfg.width,3)
                                    and v[k+'_depth_mm'].shape==(T,cfg.height,cfg.width)
                                    and v[k+'_segmentation_int32'].shape==(T,cfg.height,cfg.width)
                                    and pc[k+'_world_xyz_m'].shape==(T,cfg.sample_points,3)
                                    for k in cfg.camera_names)
    checks['optical_flow']=bool(v['front_optical_flow_rgb_estimated_px'].shape==(T,cfg.height,cfg.width,2))
    checks['tactile']=bool(tactile['pressure_2pads_PROXY'].shape ==
                          (T,2,cfg.tactile_resolution,cfg.tactile_resolution))
    checks['qa_count']=len((pth/'qa.jsonl').read_text().splitlines())==8
    checks['valid_audio']=all((pth/f).stat().st_size>1000 for f in ['spoken_instruction.wav','impact_audio_SYNTHETIC.wav'])
    checks['no_leaked_api_key']=cfg.api_key not in (pth/'task.json').read_text() if cfg.api_key else True
    dump_json(pth/'validation.json',{'all_pass':bool(all(checks.values())),'checks':checks})
    return all(checks.values()),checks


def checksum_manifest(folder):
    ans={}
    for p in sorted(Path(folder).rglob('*')):
        if p.is_file() and p.name!='_COMPLETE.json':
            h=hashlib.sha256()
            with p.open('rb') as f:
                for block in iter(lambda:f.read(2**20),b''):h.update(block)
            ans[str(p.relative_to(folder))]={'sha256':h.hexdigest(),'bytes':p.stat().st_size}
    return ans


def commit_to_drive(local,drive_destination,spec):
    """Full directory copy; completion marker is written last on Drive."""
    local=Path(local);target=Path(drive_destination)
    if target.exists() and (target/'_COMPLETE.json').exists():
        raise FileExistsError('Complete episode already exists: '+str(target))
    partial=target.with_name(target.name+'__partial')
    if partial.exists():shutil.rmtree(partial)
    partial.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(local,partial)
    source_hash=checksum_manifest(local);copied_hash=checksum_manifest(partial)
    if source_hash!=copied_hash:raise IOError('Drive copy failed checksum validation: '+spec['episode_id'])
    dump_json(partial/'_COMPLETE.json',{'dataset_version':DATASET_VERSION,
                   'episode_id':spec['episode_id'],'sha256':source_hash,'verified':True})
    if target.exists():shutil.rmtree(target)
    partial.rename(target)
    return sum(v['bytes'] for v in source_hash.values())


def generate_shard(cfg, preflight_per_task=1):
    """Idempotent resume: create 4*100 accepted episodes with per-task quotas."""
    assert cfg.owner in OWNERS and cfg.limit_per_task in range(1,101)
    import tempfile
    root=Path(cfg.output_root)/cfg.owner
    root.mkdir(parents=True,exist_ok=True)
    rows=[r for r in manifest_for_owner(cfg.owner) if int(r['episode_id'][-4:]) < cfg.limit_per_task]
    # Cover ALL four task families in the preflight before spending hours on bulk data.
    preflight=[r for r in rows if int(r['episode_id'][-4:]) < preflight_per_task]
    bulk=[r for r in rows if int(r['episode_id'][-4:]) >= preflight_per_task]
    run_order=preflight+bulk
    dump_json(root/'task_manifest.json',{'dataset_version':DATASET_VERSION,
                          'owner':cfg.owner,'rows':rows,'schema':'shared Omni-EE v1'})
    logpath=root/'generation_attempts.jsonl'
    count=0
    for row in run_order:
        dest=root/'episodes'/row['episode_id']
        if (dest/'_COMPLETE.json').exists():
            count+=1;continue
        accepted=False
        for attempt in range(cfg.max_attempts):
            seed=int(row['attempt_seed']+attempt*10000007)
            spec=sample_spec(row,seed)
            working=Path(tempfile.mkdtemp(prefix=row['episode_id']+'_',dir='/content' if Path('/content').is_dir() else None))
            try:
                result=pybullet_episode(spec,cfg,working)
                qas=canonical_qa(spec,result)
                if cfg.enable_api:
                    api_paraphrase_batch(spec,qas,cfg)
                with (working/'qa.jsonl').open('w') as f:
                    for qi,qa in enumerate(qas):
                        qa['question_id']=f"{row['episode_id']}_Q{qi+1:02d}"
                        f.write(json.dumps(qa,ensure_ascii=False)+'\n')
                valid,checks=validate_episode(working,spec,result,cfg)
                if not valid:raise ValueError('Episode validation unsuccessful: '+str(checks))
                manifest={'dataset_version':DATASET_VERSION,'episode_id':row['episode_id'],
                          'owner':cfg.owner,'split':row['split'],'task_id':row['task_id'],
                          'seed':seed,'fps':cfg.fps,'physics_hz':cfg.physics_hz,
                          'modalities':['rgb','video','depth','segmentation','optical_flow_estimated',
                          'pointcloud','mesh','speech','impact_audio_proxy','tactile_proxy','force',
                          'time_series','joint_state','ee_pose','expert_action','qa'],
                          'record_provenance':{'speech':'espeak','impact_audio':'synthetic',
                          'tactile':'contact_or_grip_proxy','expert':'kinematic_IK',
                          'task_execution':'some_tasks_assisted_or_virtual'},
                          'action_visibility_rule':'at time t, no features from later timestamps or ground-truth action',
                          'qa_visibility_rule':'entire recorded episode allowed'}
                dump_json(working/'manifest.json',manifest)
                bytes_saved=commit_to_drive(working,dest,spec)
                print(f"OK {row['episode_id']} [{row['split']}] {bytes_saved/1e6:.1f} MB")
                with logpath.open('a') as f:f.write(json.dumps({'episode':row['episode_id'],
                   'attempt':attempt,'seed':seed,'status':'success','bytes':bytes_saved})+'\n')
                accepted=True;count+=1;break
            except Exception as e:
                detail=str(e)[:350]
                with logpath.open('a') as f:f.write(json.dumps({'episode':row['episode_id'],
                  'attempt':attempt,'seed':seed,'status':'reject','reason':detail})+'\n')
                print('REJECT',row['episode_id'],'attempt',attempt+1,detail)
            finally:
                shutil.rmtree(working,ignore_errors=True)
        if not accepted:
            raise RuntimeError(f"Cannot produce valid episode {row['episode_id']} in {cfg.max_attempts} tries; "
                               f"see {logpath}. Stopped instead of fabricating a success.")
    print(f'COMPLETED {count}/{len(rows)} episodes for {cfg.owner}; Google Drive: {root}')
    return root


def audit_saved_shard(root,expected=None):
    root=Path(root);eps=sorted((root/'episodes').glob('T*_*'))
    verified=[];issues=[];size=0
    for e in eps:
        marker=e/'_COMPLETE.json'
        if not marker.exists():issues.append((e.name,'missing completion marker'));continue
        data=json.loads(marker.read_text());paths=data['sha256'];valid=True
        for name,rec in paths.items():
            p=e/name
            if not p.exists() or p.stat().st_size!=rec['bytes']:
                issues.append((e.name,'bad size/missing '+name));valid=False;break
        if valid:verified.append(e.name);size+=sum(r['bytes'] for r in paths.values())
    print('Drive audit:',len(verified),'valid /',len(eps),'present; size GB:',round(size/1e9,3))
    print('Split counts:',{s:sum(1 for k in verified if split_of(int(k[-4:]))==s) for s,_ in SPLITS})
    if expected is not None and len(verified)!=expected:issues.append(('shard',f'expected {expected}, got {len(verified)}'))
    if issues:print('Issues:',issues[:25])
    return {'valid':len(verified),'issues':issues,'bytes':size}


## 4. Dataset configuration (400 accepted episodes, resumable)

Default is 400. To test the Colab setup with only four episodes, temporarily set `EPISODES_PER_TASK = 1`, but restore it to **100** for the real shard. The full Drive output uses a separate folder for each teammate.

In [ ]:
cfg = Config(
    owner='Lazizbek',
    output_root=DRIVE_BASE,
    fps=12,
    physics_hz=240,
    duration_s=8.0,
    width=192,
    height=144,
    camera_names=('front','overhead','side'),
    sample_points=256,
    max_attempts=14,
    api_key=API_KEY,
    api_chat_url=API_CHAT_URL,
    api_model=API_MODEL,
    enable_api=USE_API,
    limit_per_task=100,
)
assert len(manifest_for_owner(cfg.owner)) == 400
print('Owner:',cfg.owner,'| tasks:',OWNERS[cfg.owner],'| target accepted episodes:',cfg.limit_per_task*4)
print('Expected split counts:',{name:count*4 for name,count in SPLITS})
print('Output:',Path(cfg.output_root)/cfg.owner)

## 5. Generate all assigned episodes and save each one to your Drive

**This may take considerable CPU time.** It first runs one acceptance-tested episode from each task, then proceeds with the other episodes. If Colab disconnects, rerun the notebook: completed episodes are skipped. Rejected attempts are logged to `generation_attempts.jsonl`; no failed episode counts toward 400.

In [ ]:
out_root = generate_shard(cfg, preflight_per_task=1)
print('Generation finished:',out_root)

## 6. Verify Drive shard and inspect QA / EE schema

In [ ]:
summary = audit_saved_shard(Path(cfg.output_root)/cfg.owner,
                           expected=4*cfg.limit_per_task)
assert not summary['issues'], 'Some episodes failed storage or count audit; rerun generator.'
example = sorted((Path(cfg.output_root)/cfg.owner/'episodes').glob('T*_*'))[0]
import json,numpy as np
print('Example:',example)
print('Manifest:',json.loads((example/'manifest.json').read_text()))
with np.load(example/'actions.npz') as a:
    for key in a.files:print('EE/action:',key, a[key].shape)
print('First grounded QA:',json.loads((example/'qa.jsonl').read_text().splitlines()[0]))
from IPython.display import Video,display,Audio
display(Video(filename=str(example/'expert_preview.mp4'),embed=True))
display(Audio(filename=str(example/'spoken_instruction.wav')))

## 7. Handoff protocol

Give the project lead **the Google Drive folder path** printed above (or a Drive folder share), not one giant manually assembled ZIP. Each shard contains `task_manifest.json`, an `episodes/` directory, and `generation_attempts.jsonl`. Every completed episode has `_COMPLETE.json` checksums, `validation.json`, `qa.jsonl` and all modalities. **No API key** is saved.

**Training:** for EE prediction at time `t`, read observations through `t` only and use `expert_next_ee_target_world_m[t]` / `expert_next_ee_delta_from_current_world_m[t]`, with `action_valid[t]`. Treat `executed_next_ee_delta_world_m` as a *separate observed outcome*, not the commanded label. For Episode QA, the full time series may be read.

**Audit caveat:** PyBullet is not installed in the notebook author's build environment, so only notebook/source syntax and deterministic task/shard-manifest tests were run before upload. Run the four-task preflight in Colab before committing to the full 400.